# บทที่ 1 — สัญญาณสั่นสะเทือนบอกอะไรเราได้

<sub>บทเรียนที่ 1 จาก 8 &nbsp;·&nbsp; [สารบัญ](README.md) · [บทที่ 2 →](02_feature_extraction.ipynb)</sub>

## เป้าหมายของบทนี้

เมื่อจบบทนี้คุณจะ:

- เปิดไฟล์ข้อมูลดิบของ IMS แล้วรู้ว่าข้างในมีอะไร
- พล็อตสัญญาณสั่นและอ่านมันออก
- เห็นด้วยตาตัวเองว่าลูกปืนที่กำลังพัง สัญญาณต่างจากลูกปืนปกติอย่างไร
- แปลงสัญญาณเป็นสเปกตรัมความถี่ด้วย FFT และรู้ว่าทำไปทำไม

> **บทนี้ต้องมีข้อมูลดิบ** — ถ้ายังไม่ได้โหลด ให้ทำตาม `README.md` หัวข้อ *การติดตั้ง → ขั้นตอนที่ 3* ก่อน

---

## 1.1 ปัญหาที่เรากำลังแก้

ลูกปืน (bearing) คือชิ้นส่วนที่อยู่ระหว่างเพลาที่หมุนกับตัวเรือนที่อยู่นิ่ง
เครื่องจักรหมุนแทบทุกชนิดมีมัน และมันคือชิ้นส่วนที่พังบ่อยที่สุด

ปัญหาคือมันมักทำงาน**ปกติดีเกือบตลอดอายุ** แล้วทรุดลงเร็วมากในช่วงท้าย
ถ้ารอจนได้ยินเสียงผิดปกติก็มักสายเกินไปแล้ว

**คำถามของโปรเจกต์นี้:** เราจะรู้ล่วงหน้าได้ไหมว่าลูกปืนกำลังจะพัง
โดยดูจากการสั่นสะเทือนที่วัดได้?

## 1.2 ข้อมูลที่เราจะใช้

**IMS Bearing Dataset** จาก University of Cincinnati เป็นการทดลองแบบ run-to-failure —
เดินเครื่องจริงจนลูกปืนพังจริง ไม่ใช่การจำลอง

| รายการ | ค่า |
|---|---|
| ลูกปืนที่วัด | 4 ตัวบนเพลาเดียวกัน |
| ความเร็วรอบ | 2,000 rpm คงที่ |
| ภาระ | ~6,000 ปอนด์ คงที่ |
| อัตราสุ่มสัญญาณ | 20,000 ครั้ง/วินาที (20 kHz) |
| แต่ละไฟล์ | บันทึก 1 วินาที → 20,480 แถว × 4 คอลัมน์ |
| ความถี่การบันทึก | ทุก 10 นาที |
| จำนวนไฟล์ | 984 ไฟล์ = 6.8 วัน |

เมื่อจบการทดลอง: **Bearing 1 วงแหวนนอกแตก (outer race failure)** · Bearing 2, 3, 4 ยังปกติดี

มาเปิดดูของจริงกัน

In [ ]:
# ── ตั้งค่าให้ notebook มองเห็นโค้ดใน src/ ──
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# ── หาฟอนต์ที่แสดงภาษาไทยได้ ไม่งั้นข้อความในกราฟจะกลายเป็นสี่เหลี่ยม ──
_installed = {f.name for f in fm.fontManager.ttflist}
for _f in ["Noto Sans Thai", "Leelawadee UI", "Tahoma", "TH Sarabun New", "Angsana New"]:
    if _f in _installed:
        plt.rcParams["font.family"] = _f
        plt.rcParams["axes.unicode_minus"] = False    # ฟอนต์ไทยมักไม่มีเครื่องหมายลบแบบ unicode
        print("ฟอนต์กราฟ:", _f)
        break
else:
    print("[หมายเหตุ] ไม่พบฟอนต์ไทย - ข้อความไทยในกราฟอาจแสดงเป็นสี่เหลี่ยม")
    print("           Windows/macOS มักมีอยู่แล้ว ส่วน Linux ลง: sudo apt install fonts-thai-tlwg")

print("project root:", ROOT)

In [ ]:
from src.paths import resolve_data_dir

DATA_DIR = resolve_data_dir()
files = sorted(p.name for p in DATA_DIR.iterdir() if p.is_file()) if DATA_DIR.is_dir() else []

print("โฟลเดอร์ข้อมูล:", DATA_DIR)
print("จำนวนไฟล์     :", len(files))
if files:
    print("ไฟล์แรก       :", files[0])
    print("ไฟล์สุดท้าย   :", files[-1])
else:
    print("\n!! ยังไม่มีข้อมูล - ดู README หัวข้อการติดตั้งขั้นตอนที่ 3")

ชื่อไฟล์คือ **เวลาที่บันทึก** เช่น `2004.02.12.10.32.39` = 12 ก.พ. 2004 เวลา 10:32:39

การเรียงชื่อไฟล์ตามตัวอักษรจึงเท่ากับเรียงตามเวลาพอดี — สะดวกมาก

In [ ]:
from src.data_loader import load_single_file

# เปรียบเทียบสองช่วงเวลา: ต้นการทดลอง (ปกติ) กับท้ายการทดลอง (ใกล้พัง)
early = load_single_file(str(DATA_DIR / files[0]))
late  = load_single_file(str(DATA_DIR / files[-1]))

print("รูปร่างข้อมูล 1 ไฟล์:", early.shape, " (20,480 จุด x 4 ลูกปืน)")
print()
print("ค่า 5 แถวแรกของไฟล์แรก:")
print(early[:5])

## 1.3 พล็อตสัญญาณออกมาดู

ทีนี้มาดูของจริง — เอาลูกปืนที่พัง (Bearing 1, คอลัมน์ index 0) มาเทียบกันระหว่างต้นกับท้ายการทดลอง

In [ ]:
CH = 0   # Bearing 1 (index เริ่มที่ 0) - ตัวที่วงแหวนนอกแตก
N  = 2000  # แสดงแค่ 2,000 จุดแรก (= 0.1 วินาที) จะได้เห็นรายละเอียด

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharey=True)

axes[0].plot(early[:N, CH], linewidth=0.7, color="#2e7d32")
axes[0].set_title("Bearing 1 - จุดเริ่มต้นการทดลอง (ปกติ)")
axes[0].set_ylabel("ความเร่ง (g)")

axes[1].plot(late[:N, CH], linewidth=0.7, color="#c62828")
axes[1].set_title("Bearing 1 - ท้ายการทดลอง (วงแหวนนอกแตกแล้ว)")
axes[1].set_ylabel("ความเร่ง (g)")
axes[1].set_xlabel("ลำดับตัวอย่าง (20,000 จุด = 1 วินาที)")

plt.tight_layout()
plt.show()

**สังเกตสองอย่าง:**

1. **แอมพลิจูดใหญ่ขึ้นมาก** — เส้นล่างสั่นแรงกว่าเส้นบนหลายเท่า
2. **รูปร่างเปลี่ยนไป** — เส้นบนดูเหมือนสัญญาณรบกวนสุ่มทั่วไป
   ส่วนเส้นล่างมี "ยอดแหลม" โผล่ขึ้นมาเป็นจังหวะ

ข้อ 2 สำคัญกว่าข้อ 1 มาก และนี่คือหัวใจของงานนี้

**ทำไมถึงมียอดแหลม?** เพราะเมื่อผิวโลหะมีรอยแตก ทุกครั้งที่ลูกกลิ้งวิ่งผ่านรอยนั้น
จะเกิดการ**กระแทกสั้น ๆ ที่คมมาก** — เหมือนเคาะโลหะเบา ๆ เป็นจังหวะทุกรอบการหมุน

## 1.4 เทียบทั้ง 4 ลูกปืนพร้อมกัน

มีแค่ Bearing 1 ที่พัง อีกสามตัวปกติดีตลอด — มาดูว่าต่างกันแค่ไหนตอนท้ายการทดลอง

In [ ]:
labels = ["Bearing 1 (วงแหวนนอกแตก)", "Bearing 2 (ปกติ)",
          "Bearing 3 (ปกติ)", "Bearing 4 (ปกติ)"]
colors = ["#d32f2f", "#388e3c", "#1976d2", "#f57c00"]

fig, axes = plt.subplots(4, 1, figsize=(11, 8), sharex=True, sharey=True)
for ch in range(4):
    axes[ch].plot(late[:N, ch], linewidth=0.6, color=colors[ch])
    axes[ch].set_title(labels[ch], fontsize=10, loc="left")
    axes[ch].set_ylabel("g")
axes[-1].set_xlabel("ลำดับตัวอย่าง")
plt.suptitle("สัญญาณสั่นตอนท้ายการทดลอง - ทั้ง 4 ลูกปืน", y=1.00)
plt.tight_layout()
plt.show()

เห็นชัดว่าเส้นบนสุด (Bearing 1 ที่พัง) สั่นแรงและมียอดแหลมมากกว่าอีกสามเส้นอย่างชัดเจน

แต่**ตาเปล่าดูได้แค่นี้** เราต้องการตัวเลขที่วัดความต่างนี้ได้ เพื่อป้อนให้คอมพิวเตอร์
— นั่นคือเนื้อหาของบทที่ 2

## 1.5 มองสัญญาณอีกมุม — โดเมนความถี่

จนถึงตอนนี้เราดูสัญญาณใน **โดเมนเวลา** (แกน x = เวลา)
แต่มีอีกมุมที่บอกอะไรได้เยอะมาก คือ **โดเมนความถี่**

**FFT (Fast Fourier Transform)** คือเครื่องมือที่ตอบคำถามว่า
*"สัญญาณนี้ประกอบด้วยคลื่นความถี่อะไรบ้าง แต่ละความถี่แรงแค่ไหน"*

เปรียบเทียบง่าย ๆ: ถ้าสัญญาณคือเสียงเพลง โดเมนเวลาคือคลื่นเสียงที่ไมค์รับได้
ส่วน FFT คือการแยกออกมาว่าในเพลงนั้นมีโน้ตอะไรบ้าง โน้ตไหนดังแค่ไหน

**ทำไมมันสำคัญกับลูกปืน?** เพราะการกระแทกที่ "สั้นและคม" จะกระจายพลังงานไปที่**ความถี่สูง**
ดังนั้นถ้าเห็นพลังงานย่านความถี่สูงเพิ่มขึ้น = มีการกระแทกโลหะเกิดขึ้น

In [ ]:
def spectrum(signal, fs=20000):
    """แปลงสัญญาณเป็นสเปกตรัมความถี่ด้วย FFT

    คืนค่า (ความถี่ Hz, ขนาดของแต่ละความถี่)
    """
    n = len(signal)
    mag = np.abs(np.fft.rfft(signal)) / n     # rfft = ใช้เฉพาะความถี่บวก
    freq = np.fft.rfftfreq(n, d=1.0 / fs)     # แปลง index -> Hz
    return freq, mag


f_early, m_early = spectrum(early[:, CH])
f_late,  m_late  = spectrum(late[:, CH])

plt.figure(figsize=(11, 4.5))
plt.plot(f_early, m_early, linewidth=0.7, label="ต้นการทดลอง (ปกติ)", color="#2e7d32", alpha=0.85)
plt.plot(f_late,  m_late,  linewidth=0.7, label="ท้ายการทดลอง (พังแล้ว)", color="#c62828", alpha=0.85)
plt.xlabel("ความถี่ (Hz)")
plt.ylabel("ขนาด")
plt.title("สเปกตรัมความถี่ของ Bearing 1")
plt.legend()
plt.tight_layout()
plt.show()

เส้นแดงสูงกว่าเส้นเขียวเกือบทุกความถี่ แต่ที่น่าสนใจคือ**อัตราส่วน** —
พลังงานเพิ่มขึ้นที่ย่านความถี่สูงมากกว่าย่านความถี่ต่ำ

มาวัดกันจริง ๆ โดยแบ่งเป็น 3 ย่าน

In [ ]:
def band_energy(freq, mag, lo, hi):
    """พลังงานรวมในย่านความถี่ที่กำหนด"""
    sel = (freq >= lo) & (freq < hi)
    return float(np.sum(mag[sel] ** 2))


bands = [("ต่ำ  0-2.5 kHz", 0, 2500),
         ("กลาง 2.5-7.5 kHz", 2500, 7500),
         ("สูง  7.5-10 kHz", 7500, 10000)]

print(f"{'ย่านความถี่':<18} {'ปกติ':>12} {'พังแล้ว':>12} {'เพิ่มขึ้นกี่เท่า':>16}")
print("-" * 62)
for name, lo, hi in bands:
    e0 = band_energy(f_early, m_early, lo, hi)
    e1 = band_energy(f_late,  m_late,  lo, hi)
    print(f"{name:<18} {e0:>12.4f} {e1:>12.4f} {e1/e0:>15.1f}x")

**นี่คือเหตุผลที่เราจะสกัด "Band Energy" ทั้ง 3 ย่านเป็น feature ในบทถัดไป**

ตัวเลขอัตราส่วนนี้บอกเราว่าย่านความถี่ไหนไวต่อการเสื่อมที่สุด

## 🔧 ลองแก้ดู — ลองเปลี่ยนมุมมอง


ลองแก้เซลล์ด้านบนแล้วรันใหม่ ดูว่าผลเปลี่ยนไปอย่างไร:

1. เปลี่ยน `CH = 0` เป็น `CH = 1` (Bearing 2 ที่ปกติดีตลอด) —
   สัญญาณต้นกับท้ายยังต่างกันไหม? ควรต่างน้อยกว่ามาก
2. เปลี่ยน `N = 2000` เป็น `N = 20480` เพื่อดูสัญญาณเต็ม 1 วินาที
3. ลองโหลดไฟล์ตรงกลางการทดลอง เช่น `files[len(files)//2]`
   แล้วดูว่ามันอยู่ตรงไหนระหว่างปกติกับพัง

## ❓ เช็คความเข้าใจ

**1. ทำไมเราถึงบันทึกแค่ 1 วินาทีทุก ๆ 10 นาที แทนที่จะบันทึกต่อเนื่องตลอด?**

<details>
<summary>ดูเฉลย</summary>

เพราะที่ 20 kHz การบันทึกต่อเนื่อง 7 วันจะได้ข้อมูลราว 12 พันล้านจุดต่อลูกปืน ซึ่งเก็บและประมวลผลไม่ไหว การสุ่มตัวอย่าง 1 วินาทีทุก 10 นาทีก็เพียงพอแล้ว เพราะการเสื่อมของลูกปืนเกิดขึ้นในสเกลชั่วโมงถึงวัน ไม่ใช่วินาที

</details>

**2. ถ้าดูแค่ค่าแอมพลิจูดสูงสุด (peak) อย่างเดียว จะพอบอกได้ไหมว่าลูกปืนกำลังเสื่อม?**

<details>
<summary>ดูเฉลย</summary>

บอกได้บ้างแต่ช้าเกินไป เพราะแอมพลิจูดจะเพิ่มชัดเจนก็ต่อเมื่อความเสียหายลุกลามแล้ว ส่วนตัวชี้วัดที่จับ 'ความแหลม' ของสัญญาณ (เช่น Kurtosis) จะตอบสนองตั้งแต่เพิ่งเริ่มมีรอยแตก — เราจึงใช้หลายตัวชี้วัดพร้อมกัน

</details>

**3. FFT บอกอะไรที่การดูสัญญาณในโดเมนเวลาบอกไม่ได้?**

<details>
<summary>ดูเฉลย</summary>

บอกว่าพลังงานการสั่นกระจายอยู่ที่ความถี่ไหนบ้าง การกระแทกสั้น ๆ จากรอยแตกจะปรากฏเป็น พลังงานย่านความถี่สูง ซึ่งในโดเมนเวลาจะกลืนไปกับสัญญาณรบกวนจนแยกยาก

</details>

---

## สรุปบทนี้

- ข้อมูล IMS คือการเดินเครื่องจริงจนลูกปืนพัง 984 จุดเวลา ครอบคลุม 6.8 วัน
- ลูกปืนที่เสื่อมจะสั่นแรงขึ้น และที่สำคัญกว่าคือมี 'ยอดแหลม' จากการกระแทก
- FFT ช่วยให้เห็นว่าพลังงานย้ายไปอยู่ย่านความถี่สูงเมื่อเกิดความเสียหาย
- แต่ข้อมูลดิบ 20,480 จุดต่อไฟล์ใหญ่เกินกว่าจะป้อนโมเดลตรง ๆ

[สารบัญ](README.md) &nbsp;·&nbsp; **[บทที่ 2 — สกัด Features →](02_feature_extraction.ipynb)**